# 모기 비행 궤적 예측 — GRU + Transformer + 물리 디코더

**Public 0.7022 / Private 0.700**

평가지표 R-Hit@1cm이 1cm 경계 안에 드느냐로 갈리는 이진 지표라는 점에 착안해,
좌표를 직접 예측하는 대신 운동 방정식의 계수를 학습하고 경계 안쪽으로 점을 밀어넣는 데 집중한 솔루션입니다.

**구성**
1. 설정 & 재현성
2. 데이터 로딩
3. 피처 엔지니어링
4. 물리 디코더 & 손실
5. 모델 정의 (GRU + Transformer)
6. Dataset & 피처 생성
7. 학습 루프 (사전학습 → 파인튜닝 → 10-Fold)
8. 후처리 (Weighted LGBM 잔차 보정)
9. 제출 파일 생성

> 데이터 경로가 Kaggle Dataset 기준(`/kaggle/input/...`)입니다. Dacon 데이터로 실행할 때는 1번 셀의 `DATA_DIR`를 수정하세요.


## 1. 설정 & 재현성

경로, 하이퍼파라미터, Transformer 설정, seed 고정(완전 재현)을 정의합니다.

In [ ]:
"""
GRU + Transformer 하이브리드 모델
=====================================
구조:
  1. GRU:         시퀀스 → hidden states (모든 timestep)
  2. Transformer: GRU hidden states → Self-Attention
  3. Pooling:     CLS 토큰으로 집약
  4. Head:        6 파라미터 예측
  5. 물리공식:    운동 방정식으로 좌표 디코딩
"""

from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold

DATA_DIR   = Path('/kaggle/input/datasets/woojinroh/mos-data')
TRAIN_DIR  = DATA_DIR / 'train'
TEST_DIR   = DATA_DIR / 'test'
LABEL_PATH = DATA_DIR / 'train_labels.csv'
SUB_PATH   = DATA_DIR / 'sample_submission.csv'

HIDDEN     = 128
N_LAYERS   = 2
DROPOUT    = 0.08
BATCH      = 512
PRE_EPOCHS = 40
FINE_EPOCHS= 60
LR         = 1e-3
FINE_LR    = LR * 0.20
N_SPLITS   = 10
R_HIT      = 0.01
EPS        = 1e-8
PATIENCE   = 15
MIN_EPOCHS = 5

# Transformer 하이퍼파라미터
N_HEADS    = 4    # Attention head 수 (HIDDEN=128, 128/4=32 per head)
N_TF_LAYERS= 2    # Transformer layer 수
TF_DIM     = 256  # Transformer FF dim

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

# 재현성 확보
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)
print("시드 고정: 42 (완전 재현성)")

## 2. 데이터 로딩

train/test 궤적 시계열(x, y, z)과 라벨을 읽어옵니다.

In [ ]:
def read_xyz(path):
    df = pd.read_csv(path).sort_values('timestep_ms')
    return df[['x','y','z']].to_numpy(dtype=np.float32)

def load_all(folder, pattern):
    files = sorted(folder.glob(pattern))
    return np.stack([read_xyz(f) for f in files], axis=0), [f.stem for f in files]

print("데이터 로딩...")
train_labels = pd.read_csv(LABEL_PATH)
sample_sub   = pd.read_csv(SUB_PATH)
X_train_raw, train_ids = load_all(TRAIN_DIR, 'TRAIN_*.csv')
X_test_raw,  test_ids  = load_all(TEST_DIR,  'TEST_*.csv')
train_labels = train_labels.set_index('id').loc[train_ids].reset_index()
Y_train = train_labels[['x','y','z']].to_numpy(dtype=np.float32)
print(f"Train: {X_train_raw.shape}")

## 3. 피처 엔지니어링

속도·곡률·z축 성분 등 6-step 시퀀스 피처와, 물리 디코딩에 필요한 컨텍스트(p0, 속도, 접선/법선 가속 등)를 만듭니다.

In [ ]:
def make_seq_features(x, end_idx):
    feats = []
    indices = list(range(max(3, end_idx-5), end_idx+1))
    if len(indices) < 6:
        indices = [indices[0]] * (6-len(indices)) + indices
    for idx in indices:
        d1=x[:,idx]-x[:,idx-1]; d2=x[:,idx-1]-x[:,idx-2]
        d3=x[:,idx-2]-x[:,idx-3] if idx>=3 else d2
        acc=d1-d2; prev_acc=d2-d3
        tang=d1/(np.linalg.norm(d1,axis=1,keepdims=True)+EPS)
        spd=np.linalg.norm(d1,axis=1,keepdims=True)
        pspd=np.linalg.norm(d2,axis=1,keepdims=True)
        ap=np.sum(acc*tang,axis=1,keepdims=True)
        aperp=acc-ap*tang
        pnorm=np.linalg.norm(aperp,axis=1,keepdims=True)
        jnorm=np.linalg.norm(acc-prev_acc,axis=1,keepdims=True)
        tcos=np.sum(d1*d2,axis=1,keepdims=True)/(spd*pspd+EPS)
        curv=pnorm/(spd+EPS)
        z_spd=d1[:,2:3]/(spd+EPS); z_acc=acc[:,2:3]/(spd+EPS)
        cross_z=(tang[:,0:1]*d2[:,1:2]-tang[:,1:2]*d2[:,0:1])/(pspd+EPS)
        feats.append(np.concatenate([
            spd,pspd/(spd+EPS),ap/(spd+EPS),pnorm/(spd+EPS),
            tcos,curv,z_spd,z_acc,cross_z,
            jnorm/(spd+EPS),np.ones((len(x),1),dtype=np.float32)],axis=1))
    N=len(x)
    early_end=min(5,end_idx)
    ev=x[:,1:early_end+1]-x[:,0:early_end]
    espd=np.linalg.norm(ev,axis=2).mean(axis=1,keepdims=True) \
         if ev.shape[1]>0 else np.zeros((N,1),dtype=np.float32)
    ls=max(1,end_idx-3); lv=x[:,ls:end_idx]-x[:,ls-1:end_idx-1]
    lspd=np.linalg.norm(lv,axis=2).mean(axis=1,keepdims=True) \
         if lv.shape[1]>0 else np.zeros((N,1),dtype=np.float32)
    spd_trend=lspd-espd
    rs=max(0,end_idx-3)
    od=x[:,end_idx]-x[:,0]; rd=x[:,end_idx]-x[:,rs]
    od=od/(np.linalg.norm(od,axis=1,keepdims=True)+EPS)
    rd=rd/(np.linalg.norm(rd,axis=1,keepdims=True)+EPS)
    tsig=np.sum(od*rd,axis=1,keepdims=True)
    zr=max(0,end_idx-4); zt=(x[:,end_idx,2]-x[:,zr,2]).reshape(-1,1)
    z_lspd=np.abs(lv[:,:,2]).mean(axis=1,keepdims=True) \
           if lv.shape[1]>0 else np.zeros((N,1),dtype=np.float32)
    gf=np.concatenate([spd_trend,tsig,zt,espd,lspd,z_lspd],axis=1).astype(np.float32)
    base=np.stack(feats,axis=1)
    gf_e=np.repeat(gf[:,np.newaxis,:],6,axis=1)
    return np.concatenate([base,gf_e],axis=2).astype(np.float32)

def extract_physics_ctx(x, end_idx):
    p0=x[:,end_idx]; d1v=x[:,end_idx]-x[:,end_idx-1]
    d2v=x[:,end_idx-1]-x[:,end_idx-2]
    d3v=x[:,end_idx-2]-x[:,end_idx-3] if end_idx>=3 else d2v
    acc=d1v-d2v; prev_acc=d2v-d3v; jerk=acc-prev_acc
    tang=d1v/(np.linalg.norm(d1v,axis=1,keepdims=True)+EPS)
    ap=np.sum(acc*tang,axis=1,keepdims=True)*tang; aperp=acc-ap
    return (p0.astype(np.float32), d1v.astype(np.float32),
            ap.astype(np.float32), aperp.astype(np.float32),
            d2v.astype(np.float32), jerk.astype(np.float32))

def make_target_params(x, end_idx, y, horizon=2):
    p0=x[:,end_idx]; d1v=x[:,end_idx]-x[:,end_idx-1]
    d2v=x[:,end_idx-1]-x[:,end_idx-2]; acc=d1v-d2v
    tang=d1v/(np.linalg.norm(d1v,axis=1,keepdims=True)+EPS)
    ap=np.sum(acc*tang,axis=1,keepdims=True)*tang; aperp=acc-ap
    spd=np.linalg.norm(d1v,axis=1,keepdims=True)
    delta=y-p0; vs=horizon/2.0; as_=(horizon/2.0)**2
    delta_par=np.sum(delta*tang,axis=1,keepdims=True)
    perp_vec=delta-delta_par*tang
    perp_mag=np.linalg.norm(perp_vec,axis=1,keepdims=True)
    cross_z=(tang[:,0:1]*delta[:,1:2]-tang[:,1:2]*delta[:,0:1])
    perp_signed=perp_mag*np.sign(cross_z)
    ap_mag=np.linalg.norm(ap,axis=1,keepdims=True)
    aperp_mag=np.linalg.norm(aperp,axis=1,keepdims=True)
    d1s=np.clip(delta_par/(vs*spd+EPS), 0.0, 3.0)
    pars=np.clip((delta_par-d1s*vs*spd)/(as_*ap_mag+EPS), -2.0, 2.0)
    perps=np.clip(perp_signed/(as_*(aperp_mag+EPS)), -8.0, 8.0)
    ts=np.ones_like(d1s)
    d2v_=x[:,end_idx-1]-x[:,end_idx-2]
    d2_mag=np.linalg.norm(d2v_,axis=1,keepdims=True)
    d2s=np.clip(delta_par/(vs*d2_mag+EPS)*0.3, -1.0, 1.0)
    d3v_=x[:,end_idx-2]-x[:,end_idx-3] if end_idx>=3 else d2v_
    jerk_vec=(d1v-d2v_)-(d2v_-d3v_)
    jerk_mag=np.linalg.norm(jerk_vec,axis=1,keepdims=True)
    jerks=np.clip(delta_par/(as_*jerk_mag+EPS)*0.1, -1.0, 1.0)
    return np.concatenate([d1s,pars,perps,ts,d2s,jerks],axis=1).astype(np.float32)

## 4. 물리 디코더 & 손실

신경망이 출력한 6개 계수를 운동 방정식에 넣어 좌표를 계산하고, 평균 거리가 아니라 **Hit 여부를 직접 최적화**하는 손실을 정의합니다.

In [ ]:
def physics_predict_torch(p0, d1v, ap, aperp, params, horizon=2,
                           d2v=None, jerk=None):
    d1s  = params[:, 0:1]; pars = params[:, 1:2]
    perps= params[:, 2:3]; ts   = params[:, 3:4]
    d2s  = params[:, 4:5] if params.shape[1] > 4 else torch.zeros_like(d1s)
    jerks= params[:, 5:6] if params.shape[1] > 5 else torch.zeros_like(d1s)
    vs   = horizon / 2.0 * ts
    as_  = (horizon / 2.0) ** 2 * ts ** 2
    pred = p0 + d1s*vs*d1v + pars*as_*ap + perps*as_*aperp
    if d2v  is not None: pred = pred + d2s*vs*d2v
    if jerk is not None: pred = pred + jerks*as_*jerk
    return pred

def adaptive_loss(pred_params, p0, d1v, ap, aperp, y_true, d2v=None, jerk=None):
    pred_xyz = physics_predict_torch(p0,d1v,ap,aperp,pred_params,d2v=d2v,jerk=jerk)
    err      = torch.norm(pred_xyz-y_true, dim=1)
    huber    = F.huber_loss(err, torch.zeros_like(err), delta=0.015)
    hit_prob = torch.sigmoid(-(err-R_HIT)/0.003)
    hit_loss = -hit_prob.mean()
    d1s      = pred_params[:,0]
    param_reg= (torch.relu(d1s-2.5)**2+torch.relu(0.3-d1s)**2).mean()
    return huber + 0.15*hit_loss + 0.01*param_reg

## 5. 모델 정의 — GRU + Transformer

GRU가 시간 순서 의존성을 인코딩하고, CLS 토큰을 붙인 Transformer가 급회전 직전 같은 결정적 시점에 attention을 모읍니다. CLS 토큰을 궤적 요약 벡터로 써서 물리 파라미터 6개를 출력합니다.

In [ ]:
class GRUTransformerPhysics(nn.Module):
    """
    1. GRU: 시퀀스 처리 (시간 순서 의존성)
       seq (B,6,D) → hidden states (B,6,H)
    2. CLS 토큰 추가: [CLS, h1, ..., h6] → (B, 7, H)
    3. Transformer Encoder: Self-Attention으로 중요 timestep 강조
    4. CLS 토큰 추출 → 물리 파라미터 6개 예측
    """
    def __init__(self, seq_dim, hidden):
        super().__init__()
        # 1. GRU
        self.gru = nn.GRU(seq_dim, hidden, num_layers=N_LAYERS,
                           batch_first=True, dropout=DROPOUT)
        # 2. CLS 토큰 (학습 가능)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, hidden))
        nn.init.normal_(self.cls_token, std=0.02)
        # 3. Transformer Encoder
        tf_layer = nn.TransformerEncoderLayer(
            d_model=hidden, nhead=N_HEADS, dim_feedforward=TF_DIM,
            dropout=DROPOUT, activation='gelu', batch_first=True,
            norm_first=True)   # Pre-LN: 더 안정적인 학습
        self.transformer = nn.TransformerEncoder(tf_layer, num_layers=N_TF_LAYERS)
        self.ctx_norm    = nn.LayerNorm(hidden)
        # 4. 파라미터 head
        self.shared = nn.Sequential(
            nn.Linear(hidden, hidden), nn.SiLU(), nn.Dropout(DROPOUT),
            nn.Linear(hidden, hidden//2), nn.SiLU())
        self.head_d1   = nn.Linear(hidden//2, 1)
        self.head_par  = nn.Linear(hidden//2, 1)
        self.head_perp = nn.Linear(hidden//2, 1)
        self.head_ts   = nn.Linear(hidden//2, 1)
        self.head_d2   = nn.Linear(hidden//2, 1)
        self.head_jerk = nn.Linear(hidden//2, 1)

    def forward(self, seq):
        B = seq.size(0)
        gru_out, _ = self.gru(seq)                     # (B,6,H)
        cls = self.cls_token.expand(B, -1, -1)
        tf_in = torch.cat([cls, gru_out], dim=1)       # (B,7,H)
        tf_out = self.transformer(tf_in)               # (B,7,H)
        ctx  = self.ctx_norm(tf_out[:, 0, :])          # CLS 토큰
        feat = self.shared(ctx)
        d1s  = F.softplus(self.head_d1(feat))
        pars = torch.tanh(self.head_par(feat))  * 2.0
        perps= torch.tanh(self.head_perp(feat)) * 8.0
        ts   = torch.sigmoid(self.head_ts(feat)) * 0.8 + 0.6
        d2s  = torch.tanh(self.head_d2(feat))   * 1.0
        jerks= torch.tanh(self.head_jerk(feat)) * 1.0
        return torch.cat([d1s,pars,perps,ts,d2s,jerks], dim=1)

## 6. Dataset & 피처 생성

Dataset 클래스를 정의하고, 본 학습용 피처와 커리큘럼 사전학습용 데이터(궤적 중간 시점을 h=1,2로 잘라 대량 생성)를 만든 뒤 정규화합니다.

In [ ]:
class AdaptDS(Dataset):
    def __init__(self, seq, p0, d1v, ap, aperp, y, d2v=None, jerk=None):
        self.seq=torch.tensor(seq,dtype=torch.float32)
        self.p0=torch.tensor(p0,dtype=torch.float32)
        self.d1v=torch.tensor(d1v,dtype=torch.float32)
        self.ap=torch.tensor(ap,dtype=torch.float32)
        self.aperp=torch.tensor(aperp,dtype=torch.float32)
        self.y=torch.tensor(y,dtype=torch.float32)
        self.d2v=torch.tensor(d2v,dtype=torch.float32) if d2v is not None \
                  else torch.zeros_like(self.d1v)
        self.jerk=torch.tensor(jerk,dtype=torch.float32) if jerk is not None \
                   else torch.zeros_like(self.d1v)
    def __len__(self): return len(self.seq)
    def __getitem__(self,i):
        return (self.seq[i],self.p0[i],self.d1v[i],
                self.ap[i],self.aperp[i],self.y[i],
                self.d2v[i],self.jerk[i])

class TestAdaptDS(Dataset):
    def __init__(self, seq, p0, d1v, ap, aperp, d2v=None, jerk=None):
        self.seq=torch.tensor(seq,dtype=torch.float32)
        self.p0=torch.tensor(p0,dtype=torch.float32)
        self.d1v=torch.tensor(d1v,dtype=torch.float32)
        self.ap=torch.tensor(ap,dtype=torch.float32)
        self.aperp=torch.tensor(aperp,dtype=torch.float32)
        self.d2v=torch.tensor(d2v,dtype=torch.float32) if d2v is not None \
                  else torch.zeros_like(self.d1v)
        self.jerk=torch.tensor(jerk,dtype=torch.float32) if jerk is not None \
                   else torch.zeros_like(self.d1v)
    def __len__(self): return len(self.seq)
    def __getitem__(self,i):
        return (self.seq[i],self.p0[i],self.d1v[i],
                self.ap[i],self.aperp[i],
                self.d2v[i],self.jerk[i])

print("\n피처 생성...")
end_idx = X_train_raw.shape[1]-1
SEQ_TRN = make_seq_features(X_train_raw, end_idx)
P0_TRN, D1V_TRN, AP_TRN, APERP_TRN, D2V_TRN, JERK_TRN = \
    extract_physics_ctx(X_train_raw, end_idx)
SEQ_TST = make_seq_features(X_test_raw, X_test_raw.shape[1]-1)
P0_TST, D1V_TST, AP_TST, APERP_TST, D2V_TST, JERK_TST = \
    extract_physics_ctx(X_test_raw, X_test_raw.shape[1]-1)

print("사전학습 데이터 생성...")
pre_seqs,pre_p0s,pre_d1vs=[],[],[]
pre_aps,pre_aperps,pre_ys=[],[],[]
pre_d2vs,pre_jerks=[],[]
T=X_train_raw.shape[1]
for h in [1,2]:
    for ei in range(3,T-h):
        pre_seqs.append(make_seq_features(X_train_raw,ei))
        p0,d1v,ap,aperp,d2v,jerk=extract_physics_ctx(X_train_raw,ei)
        y_h=X_train_raw[:,ei+h]
        pre_p0s.append(p0); pre_d1vs.append(d1v)
        pre_aps.append(ap); pre_aperps.append(aperp)
        pre_d2vs.append(d2v); pre_jerks.append(jerk)
        pre_ys.append(y_h)

PRE_SEQ=np.vstack(pre_seqs).astype(np.float32)
PRE_P0=np.vstack(pre_p0s).astype(np.float32)
PRE_D1V=np.vstack(pre_d1vs).astype(np.float32)
PRE_AP=np.vstack(pre_aps).astype(np.float32)
PRE_APERP=np.vstack(pre_aperps).astype(np.float32)
PRE_D2V=np.vstack(pre_d2vs).astype(np.float32)
PRE_JERK=np.vstack(pre_jerks).astype(np.float32)
PRE_Y=np.vstack(pre_ys).astype(np.float32)
print(f"사전학습: {PRE_SEQ.shape[0]}개")

seq_mean=np.vstack([PRE_SEQ,SEQ_TRN]).reshape(-1,PRE_SEQ.shape[-1]).mean(0)
seq_std=np.vstack([PRE_SEQ,SEQ_TRN]).reshape(-1,PRE_SEQ.shape[-1]).std(0)+1e-6
def ns(x): return ((x-seq_mean)/seq_std).astype(np.float32)
PRE_SEQ_N=ns(PRE_SEQ); SEQ_TRN_N=ns(SEQ_TRN); SEQ_TST_N=ns(SEQ_TST)
SEQ_DIM=PRE_SEQ_N.shape[-1]
print(f"피처 차원: {SEQ_DIM}")

def hit_rate(pred, true):
    d=np.linalg.norm(pred-true,axis=1)
    return float(np.mean(d<=R_HIT)), float(d.mean())

## 7. 학습 루프 — 사전학습 → 파인튜닝 → 10-Fold

커리큘럼 사전학습(40ep)으로 표현력을 확보하고 실제 타깃으로 파인튜닝(60ep)한 뒤, 10-Fold로 학습한 모델들의 예측을 평균합니다. OOF 예측도 함께 저장해 후처리에 사용합니다.

In [ ]:
kf=KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
test_preds=[]; fold_hits=[]
oof_preds=np.zeros((len(SEQ_TRN_N),3),dtype=np.float32)

pre_ds=AdaptDS(PRE_SEQ_N,PRE_P0,PRE_D1V,PRE_AP,PRE_APERP,PRE_Y)
pre_ld=DataLoader(pre_ds,batch_size=BATCH,shuffle=True,num_workers=0)

print("\n학습 시작!")
print("="*70)

for fold,(tr_idx,va_idx) in enumerate(kf.split(SEQ_TRN_N)):
    print(f"\nFold {fold+1}/{N_SPLITS}")

    va_ds=TestAdaptDS(SEQ_TRN_N[va_idx],P0_TRN[va_idx],D1V_TRN[va_idx],
                      AP_TRN[va_idx],APERP_TRN[va_idx],D2V_TRN[va_idx],JERK_TRN[va_idx])
    tr_ds=AdaptDS(SEQ_TRN_N[tr_idx],P0_TRN[tr_idx],D1V_TRN[tr_idx],
                  AP_TRN[tr_idx],APERP_TRN[tr_idx],Y_train[tr_idx],
                  D2V_TRN[tr_idx],JERK_TRN[tr_idx])
    te_ds=TestAdaptDS(SEQ_TST_N,P0_TST,D1V_TST,AP_TST,APERP_TST,D2V_TST,JERK_TST)

    va_ld=DataLoader(va_ds,batch_size=BATCH,shuffle=False,num_workers=0)
    tr_ld=DataLoader(tr_ds,batch_size=BATCH,shuffle=True,num_workers=0)
    te_ld=DataLoader(te_ds,batch_size=BATCH,shuffle=False,num_workers=0)

    model=GRUTransformerPhysics(SEQ_DIM,HIDDEN).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
    # Transformer용 Warmup + CosineAnnealing
    warmup_epochs = 5
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return float(epoch+1) / warmup_epochs
        progress = (epoch - warmup_epochs) / (PRE_EPOCHS - warmup_epochs)
        return 0.05 + 0.95 * 0.5 * (1 + np.cos(np.pi * progress))
    sched=torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    best_hit=-1.0; best_state=None; no_imp=0

    print("  [사전학습]")
    for epoch in range(1,PRE_EPOCHS+1):
        model.train()
        for seq,p0,d1v,ap,aperp,y,d2v,jerk in pre_ld:
            seq,p0,d1v,ap,aperp,y,d2v,jerk=[t.to(DEVICE) for t in
                                              [seq,p0,d1v,ap,aperp,y,d2v,jerk]]
            opt.zero_grad()
            adaptive_loss(model(seq),p0,d1v,ap,aperp,y).backward()
            nn.utils.clip_grad_norm_(model.parameters(),2.0)
            opt.step()
        sched.step()

        model.eval(); preds=[]
        with torch.no_grad():
            for seq,p0,d1v,ap,aperp,d2v,jerk in va_ld:
                seq,p0,d1v,ap,aperp,d2v,jerk=[t.to(DEVICE) for t in
                                                [seq,p0,d1v,ap,aperp,d2v,jerk]]
                preds.append(physics_predict_torch(
                    p0,d1v,ap,aperp,model(seq),d2v=d2v,jerk=jerk).cpu().numpy())
        h,dm=hit_rate(np.vstack(preds),Y_train[va_idx])
        print(f"  [Pre] Ep {epoch:03d} | Hit:{h:.4f} | Dist:{dm*100:.4f}cm")
        if h>best_hit:
            best_hit=h
            best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}
            no_imp=0; print(f"    ✓ Best! ({best_hit:.4f})")
        else:
            no_imp+=1
            if no_imp>=PATIENCE and epoch>=MIN_EPOCHS:
                print("    Early stopping"); break

    pre_best={k:v.clone() for k,v in best_state.items()}

    print("  [파인튜닝]")
    model.load_state_dict(best_state)
    opt_fi=torch.optim.AdamW(model.parameters(),lr=FINE_LR,weight_decay=1e-4)
    sched_fi=torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt_fi,mode='max',patience=8,factor=0.5)
    best_hit_fi=best_hit; no_imp=0

    for epoch in range(1,FINE_EPOCHS+1):
        model.train()
        for seq,p0,d1v,ap,aperp,y,d2v,jerk in tr_ld:
            seq,p0,d1v,ap,aperp,y,d2v,jerk=[t.to(DEVICE) for t in
                                              [seq,p0,d1v,ap,aperp,y,d2v,jerk]]
            opt_fi.zero_grad()
            adaptive_loss(model(seq),p0,d1v,ap,aperp,y,d2v,jerk).backward()
            nn.utils.clip_grad_norm_(model.parameters(),2.0)
            opt_fi.step()

        model.eval(); preds=[]
        with torch.no_grad():
            for seq,p0,d1v,ap,aperp,d2v,jerk in va_ld:
                seq,p0,d1v,ap,aperp,d2v,jerk=[t.to(DEVICE) for t in
                                                [seq,p0,d1v,ap,aperp,d2v,jerk]]
                preds.append(physics_predict_torch(
                    p0,d1v,ap,aperp,model(seq),d2v=d2v,jerk=jerk).cpu().numpy())
        h,dm=hit_rate(np.vstack(preds),Y_train[va_idx]); sched_fi.step(h)
        print(f"  [Fine] Ep {epoch:03d} | Hit:{h:.4f} | Dist:{dm*100:.4f}cm")
        if h>best_hit_fi:
            best_hit_fi=h
            best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}
            no_imp=0; print(f"    ✓ Best! ({best_hit_fi:.4f})")
        else:
            no_imp+=1
            if no_imp>=20: print("    Early stopping"); break

    if best_hit_fi<best_hit:
        best_state=pre_best; best_hit_fi=best_hit

    fold_hits.append(best_hit_fi)
    print(f"Fold {fold+1} 완료 | Best: {best_hit_fi:.4f}")

    model.load_state_dict(best_state); model.eval()
    te_p=[]; oof_p=[]
    with torch.no_grad():
        for seq,p0,d1v,ap,aperp,d2v,jerk in te_ld:
            seq,p0,d1v,ap,aperp,d2v,jerk=[t.to(DEVICE) for t in
                                            [seq,p0,d1v,ap,aperp,d2v,jerk]]
            te_p.append(physics_predict_torch(
                p0,d1v,ap,aperp,model(seq),d2v=d2v,jerk=jerk).cpu().numpy())
        for seq,p0,d1v,ap,aperp,d2v,jerk in va_ld:
            seq,p0,d1v,ap,aperp,d2v,jerk=[t.to(DEVICE) for t in
                                            [seq,p0,d1v,ap,aperp,d2v,jerk]]
            oof_p.append(physics_predict_torch(
                p0,d1v,ap,aperp,model(seq),d2v=d2v,jerk=jerk).cpu().numpy())
    test_preds.append(np.vstack(te_p))
    oof_preds[va_idx]=np.vstack(oof_p)

print("\n"+"="*70)
for i,h in enumerate(fold_hits): print(f"  Fold {i+1}: {h:.4f}")
print(f"  평균: {np.mean(fold_hits):.4f}")

oof_h,oof_d=hit_rate(oof_preds,Y_train)
print(f"OOF: {oof_h:.4f} | 거리: {oof_d*100:.4f}cm")

## 8. 후처리 — Weighted LGBM 잔차 보정

딥러닝 구조는 그대로 두고, OOF 잔차를 LGBM이 학습하되 0.01 경계 주변 샘플에 더 큰 가중치를 줘서 경계 밖 점을 안쪽으로 밉니다. 과보정 방지를 위해 보정폭은 축당 ±0.0035로 제한합니다.

> **참고:** alpha 후보(0.00~0.30)를 모두 제출 파일로 만들어 두고, 최종 제출은 OOF가 아닌 Public LB를 보고 0.14를 선택했습니다. 결과적으로 Public 0.7022 → Private 0.700으로 하락했는데, OOF 기준을 믿고 더 보수적인 alpha를 골랐어야 했습니다.

In [ ]:
# 최종 제출: 원본 10-fold 전체 평균(all) + LightGBM Residual
final_all = np.mean(test_preds, axis=0)

# 참고용: good selection도 함께 저장
thr = np.mean(fold_hits) - 0.005
good = [i for i, h in enumerate(fold_hits) if h >= thr]
final_good = np.mean([test_preds[i] for i in good], axis=0)
print("All folds 사용:", list(range(1, len(test_preds)+1)))
print("Good folds 사용:", [i+1 for i in good])

def make_submission(pred, fn):
    df = pd.DataFrame({'id': test_ids, 'x': pred[:,0], 'y': pred[:,1], 'z': pred[:,2]})
    sample_sub[['id']].merge(df, on='id', how='left').to_csv(fn, index=False)
    print(f"✓ {fn}")

make_submission(final_all,  '/kaggle/working/submission_transformer_ep60_all.csv')
make_submission(final_good, '/kaggle/working/submission_transformer_ep60_good.csv')

# ── Weighted LGBM residual boosting ──
# 목적: 평균거리 감소가 아니라 hit boundary(0.01) 근처 샘플을 안쪽으로 밀기
USE_WEIGHTED_LGBM_RESIDUAL = True
ALPHA_CANDIDATES = [round(x, 3) for x in np.arange(0.00, 0.301, 0.01)]
RESIDUAL_CLIP = 0.0035       # 한 축당 최대 보정폭 제한
LGBM_META_SPLITS = 5

# hit boundary 가중치
HARD_WEIGHT = 1.5            # err > 0.010
BOUNDARY_WEIGHT = 2.0        # 0.010 < err < 0.013
NEAR_BOUNDARY_WEIGHT = 1.3   # 0.008 < err <= 0.010

if USE_WEIGHTED_LGBM_RESIDUAL:
    try:
        import lightgbm as lgb
    except Exception:
        import sys, subprocess
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm'])
        import lightgbm as lgb

    def build_lgbm_features(seq_n, p0, d1v, ap, aperp, d2v, jerk, base_pred):
        """딥러닝 구조는 건드리지 않고, base prediction 주변 정보만 사용."""
        move = base_pred - p0
        speed = np.linalg.norm(d1v, axis=1, keepdims=True)
        acc = ap + aperp
        acc_norm = np.linalg.norm(acc, axis=1, keepdims=True)
        aperp_norm = np.linalg.norm(aperp, axis=1, keepdims=True)
        d2_norm = np.linalg.norm(d2v, axis=1, keepdims=True)
        jerk_norm = np.linalg.norm(jerk, axis=1, keepdims=True)
        move_norm = np.linalg.norm(move, axis=1, keepdims=True)
        d1_norm = np.linalg.norm(d1v, axis=1, keepdims=True) + EPS
        move_norm_safe = move_norm + EPS
        cos_move_d1 = np.sum(move * d1v, axis=1, keepdims=True) / (move_norm_safe * d1_norm)
        cos_move_acc = np.sum(move * acc, axis=1, keepdims=True) / (move_norm_safe * (acc_norm + EPS))
        seq_flat = seq_n.reshape(seq_n.shape[0], -1)
        feats = np.concatenate([
            seq_flat, p0, d1v, ap, aperp, d2v, jerk,
            base_pred, move,
            speed, acc_norm, aperp_norm, d2_norm, jerk_norm, move_norm,
            cos_move_d1, cos_move_acc,
        ], axis=1).astype(np.float32)
        return feats

    print("\n[Weighted LGBM residual] feature 생성...")
    X_meta = build_lgbm_features(
        SEQ_TRN_N, P0_TRN, D1V_TRN, AP_TRN, APERP_TRN, D2V_TRN, JERK_TRN, oof_preds)
    X_meta_test = build_lgbm_features(
        SEQ_TST_N, P0_TST, D1V_TST, AP_TST, APERP_TST, D2V_TST, JERK_TST, final_all)
    y_res = (Y_train - oof_preds).astype(np.float32)

    # OOF error 기준 sample weight
    oof_err = np.linalg.norm(oof_preds - Y_train, axis=1)
    sample_weight = np.ones(len(oof_err), dtype=np.float32)
    near_boundary = (oof_err > 0.008) & (oof_err <= R_HIT)
    hard = oof_err > R_HIT
    boundary = (oof_err > R_HIT) & (oof_err < 0.013)
    sample_weight[near_boundary] = NEAR_BOUNDARY_WEIGHT
    sample_weight[hard] = HARD_WEIGHT
    sample_weight[boundary] = BOUNDARY_WEIGHT

    print(f"[LGBM] X_meta: {X_meta.shape}, residual: {y_res.shape}")
    print(f"[LGBM] base OOF err > 0.010: {hard.mean():.4f}")
    print(f"[LGBM] sample_weight mean: {sample_weight.mean():.4f}")

    # 얕고 규제 강한 LGBM (residual noise 암기 방지)
    lgb_params = dict(
        n_estimators=600, learning_rate=0.02, num_leaves=24, max_depth=3,
        min_child_samples=120, subsample=0.80, subsample_freq=1,
        colsample_bytree=0.80, reg_alpha=0.3, reg_lambda=2.0,
        objective='regression', random_state=42, n_jobs=-1, verbose=-1)

    # Residual 모델도 OOF로 검증
    meta_kf = KFold(n_splits=LGBM_META_SPLITS, shuffle=True, random_state=42)
    res_oof = np.zeros_like(y_res, dtype=np.float32)
    print("[LGBM] OOF residual 학습...")
    for mfold, (mtr, mva) in enumerate(meta_kf.split(X_meta), 1):
        for axis in range(3):
            model_lgb = lgb.LGBMRegressor(**lgb_params)
            model_lgb.fit(X_meta[mtr], y_res[mtr, axis], sample_weight=sample_weight[mtr])
            res_oof[mva, axis] = model_lgb.predict(X_meta[mva]).astype(np.float32)
        res_tmp = np.clip(res_oof[mva], -RESIDUAL_CLIP, RESIDUAL_CLIP)
        h_tmp, d_tmp = hit_rate(oof_preds[mva] + 0.05 * res_tmp, Y_train[mva])
        print(f"  MetaFold {mfold}/{LGBM_META_SPLITS} | alpha=0.05 OOF Hit:{h_tmp:.5f}")
    res_oof = np.clip(res_oof, -RESIDUAL_CLIP, RESIDUAL_CLIP)

    # alpha 후보별 OOF hit 계산
    rows = []
    base_h, base_d = hit_rate(oof_preds, Y_train)
    best_alpha = 0.0; best_hit = base_h; best_dist = base_d
    print("\n[LGBM] alpha별 OOF 예상 성능")
    for a in ALPHA_CANDIDATES:
        h, d = hit_rate(oof_preds + a * res_oof, Y_train)
        rows.append({'alpha': a, 'oof_hit': h, 'oof_dist_cm': d * 100})
        if (h > best_hit) or (h == best_hit and d < best_dist):
            best_alpha, best_hit, best_dist = a, h, d
    pd.DataFrame(rows).to_csv('/kaggle/working/weighted_lgbm_residual_alpha_oof_results.csv', index=False)
    print(f"[LGBM] OOF 최적 alpha: {best_alpha:.3f} | OOF Hit:{best_hit:.5f}")

    # 전체 residual 데이터로 최종 LGBM 학습 후 test residual 예측
    print("[LGBM] full residual 모델 학습 및 test 보정...")
    res_test = np.zeros_like(final_all, dtype=np.float32)
    for axis in range(3):
        model_lgb = lgb.LGBMRegressor(**lgb_params)
        model_lgb.fit(X_meta, y_res[:, axis], sample_weight=sample_weight)
        res_test[:, axis] = model_lgb.predict(X_meta_test).astype(np.float32)
    res_test = np.clip(res_test, -RESIDUAL_CLIP, RESIDUAL_CLIP)

## 9. 제출 파일 생성

alpha 후보별 제출 파일을 모두 만들고, OOF 최적 alpha와 실제 최종 제출(alpha=0.14)을 각각 저장합니다.

In [ ]:
if USE_WEIGHTED_LGBM_RESIDUAL:
    # 후보 alpha별 제출 파일 생성
    for a in ALPHA_CANDIDATES:
        pred_lgb = final_all + a * res_test
        alpha_tag = f'{a:.3f}'.replace('.', 'p')
        make_submission(pred_lgb, f'/kaggle/working/submission_weighted_lgbm_residual_alpha_{alpha_tag}.csv')

    # OOF 기준 best alpha 제출
    pred_best = final_all + best_alpha * res_test
    make_submission(pred_best, '/kaggle/working/submission_weighted_lgbm_residual_best_oof.csv')

    # 최종 제출: alpha=0.14 (Public LB 기준 선택)
    pred_final = final_all + 0.14 * res_test
    make_submission(pred_final, '/kaggle/working/submission_FINAL_alpha_0p140.csv')

print("\n완료!")